In [1]:

from vllm import LLM, SamplingParams
from vllm.steer_vectors.request import SteerVectorRequest
import json
from datasets import load_dataset
import re
from typing import Optional
import os
from pathlib import Path
from transformers import AutoProcessor
from tqdm import tqdm

In [3]:
p  = Path("steer_output_diffmean_new")
p.mkdir(exist_ok=True)

In [4]:
for dataset in ["AMBER", "DASH_B", "MME", "POPE"]:
    p.joinpath(dataset).mkdir(exist_ok=True)
    for model in ["Qwen2.5-VL-7B-Instruct", "Qwen3-VL-8B-Instruct", "InternVL3-8B-hf", "InternVL3_5-8B-HF"]:
        p.joinpath(dataset).joinpath(model).mkdir(exist_ok=True)

In [5]:
model_name = "Qwen/Qwen2.5-VL-7B-Instruct"
# model_name = "Qwen/Qwen3-VL-8B-Instruct"
# model_name = "OpenGVLab/InternVL3-8B-hf"
# model_name = "OpenGVLab/InternVL3_5-8B-HF"

llm = LLM(
    model=model_name, 
    enable_steer_vector=True, 
    enforce_eager=True, 
    tensor_parallel_size=1, 
    enable_chunked_prefill=False, 
    enable_prefix_caching=False,
    max_model_len=32768,
)

In [6]:
processor = AutoProcessor.from_pretrained(
    model_name, 
    padding_side='left'
)

In [7]:
_PATTERN = re.compile(
    r"^Is there a(?:n)? (?P<object>.+?) in the image\?$",
    re.IGNORECASE
)

def extract_object(sentence: str) -> Optional[str]:
    match = _PATTERN.match(sentence.strip())
    return match.group("object") if match else None


_PATTERN_AMBER = re.compile(
    r"^Is there a(?:n)? (?P<object>.+?) in this image\?$",
    re.IGNORECASE
)

def extract_object_amber(sentence: str) -> Optional[str]:
    match = _PATTERN_AMBER.match(sentence.strip())
    return match.group("object") if match else None

def _uses_an(obj: str) -> str:
    """Determine whether to use 'a' or 'an' before the object."""
    return "an" if obj[0].lower() in "aeiou" else "a"

def process_message(image, prompt):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    return {
        "prompt": processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            ),
        "multi_modal_data": {"image": [image]},
    }

In [8]:
_PATTERN_MME = re.compile(
    r"^Is there a(?:n)? (?P<object>.+?) in (?:the|this) image\? Please answer yes or no\.$",
    re.IGNORECASE
)

def extract_object_mme(sentence: str) -> Optional[str]:
    match = _PATTERN_MME.match(sentence.strip())
    return match.group("object") if match else None

def _uses_an(obj: str) -> str:
    """Determine whether to use 'a' or 'an' before the object."""
    return "an" if obj[0].lower() in "aeiou" else "a"

# Test
print(extract_object_mme("Is there a bed in this image? Please answer yes or no."))
print(extract_object_mme("Is there an dog apple in the image? Please answer yes or no."))


In [9]:
ds = load_dataset("lmms-lab/POPE", "default")["test"]

# ds = load_dataset("darkyarding/MME")["test"]
# ds = ds.filter(lambda example: "existence" == example["category"])

# ds = load_dataset("YanNeu/DASH-B")["test"]

# ds = load_dataset("visual-preference/AMBER")["query_discriminative_existence"]
# amber_annotations = json.load(Path("AMBER/data/annotations.json").open("r"))
# amber_annotations = [amber_annotations[sample["id"]] for sample in ds]

In [10]:
choice_layers = [[_] for _ in range(7, 13)] 
choice_layers

In [11]:
scales = [-0.8, -0.5, -0.2, 0.0, 0.8, 1.0, 1.2] # qwen25
scales = [-0.3, -0.2, -0.1, 0.0, 0.6, 0.8, 1.0] # qwen3
scales = [-0.8, -0.5, -0.2, 0.0, 0.3, 0.5, 0.7] # internvl3
scales = [-0.8, -0.5, -0.2, 0.0, 0.8, 1.5, 1.6] # internvl3_5
scales = sorted(scales)
scales

In [12]:
cnt = 0
steer_path = "qwen25vl_100_pair_diffmean_wo_norm.gguf"
# steer_path = "qwen3vl_100_pair_diffmean_wo_norm.gguf"
# steer_path = "internvl3_100_pair_diffmean_wo_norm.gguf"
# steer_path = "internvl3_5_100_pair_diffmean_wo_norm.gguf"

sampling_params = SamplingParams(
    max_tokens=512,
)

In [13]:
prompt = """Locate {a_an} {obj} in the image and identify its bounding box if it exists.""" ### This prompt is for qwen family
# prompt = """Please provide the bounding box coordinate of the region this s entence describes: <ref>{a_an} {obj}</ref>""" ### This prompt is for intern family

In [15]:
### This celll is for POPE, MME, DASH_B
batch_size = 128

dataset_name = "MME"  # or "POPE"
# dataset_name = "POPE"
# dataset_name = "DASH_B"
for target_layers in choice_layers:
    for scale in scales:
        output_file_name = f"Diffmean_{dataset_name}_from_AMBER_100_{str(target_layers)}_{str(scale)}.json"
        if (p / dataset_name / model_name.split("/")[-1] / output_file_name).exists():
            print(f"Skip {output_file_name}")
            continue
        print(f"Process {output_file_name}")
        cnt += 1
        baseline_request = SteerVectorRequest(
            output_file_name, 
            cnt, 
            steer_vector_local_path=steer_path,
            scale=scale, 
            target_layers=target_layers, 
            prefill_trigger_tokens=[-1], 
            generate_trigger_tokens=[-1]
        )
        results = []
        for st in tqdm(range(0, len(ds), batch_size)):
            messages = []
            for i in range(st, min(st + batch_size, len(ds))):
                image = ds[i]["image"]
                question = ds[i]["question"]
                question = question.replace("imange", "image")
                # obj = extract_object(question) # POPE
                obj = extract_object_mme(question) # MME
                # obj = ds[i]["object"] # DASH_B
                messages.append(process_message(image, prompt.format(a_an=_uses_an(obj), obj=obj)))
            outputs = llm.generate(messages, steer_vector_request=baseline_request, sampling_params=sampling_params)
            results.extend([output.outputs[0].text for output in outputs])
        json.dump(results, (p / dataset_name / model_name.split("/")[-1] / output_file_name).open("w"), indent=4)
        print("=" * 100)
    


In [16]:
# This cell if for AMBER, which has a different question format. We need to extract object differently and also use "query" instead of "question" as the field name.
batch_size = 128
dataset_name = "AMBER"
for target_layers in choice_layers:
    for scale in scales:
        output_file_name = f"{dataset_name}_from_AMBER_100_{str(target_layers)}_{str(scale)}.json"
        if (p / dataset_name / model_name.split("/")[-1] / output_file_name).exists():
            print(f"Skip {output_file_name}")
            continue
        print(f"Process {output_file_name}")
        cnt += 1
        baseline_request = SteerVectorRequest(
            output_file_name, 
            cnt, 
            steer_vector_local_path=steer_path,
            scale=scale, 
            target_layers=target_layers, 
            prefill_trigger_tokens=[-1], 
            generate_trigger_tokens=[-1]
        )
        results = []
        for st in tqdm(range(0, len(ds), batch_size)):
            messages = []
            for i in range(st, min(st + batch_size, len(ds))):
                image = ds[i]["image"]
                question = ds[i]["query"]
                question = question.replace("imange", "image")
                obj = extract_object_amber(question)
                messages.append(process_message(image, prompt.format(a_an=_uses_an(obj), obj=obj)))
            outputs = llm.generate(messages, steer_vector_request=baseline_request, sampling_params=sampling_params)
            results.extend([output.outputs[0].text for output in outputs])
        json.dump(results, (p / dataset_name / model_name.split("/")[-1] / output_file_name).open("w"), indent=4)
        print("=" * 100)
    
